In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames (store full tables in memory for later use)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Ensure timestamp is numeric (seconds) and consistent
for df in (metric_df, trace_df, log_df, error_df):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")

# UTC+8 timezone object (for any time operations if needed later)
tz = pytz.timezone("Asia/Shanghai")

# Helper to build compact summary for a DataFrame
def compact_summary(df, name, key_name=None):
    total_rows = int(len(df))
    if total_rows == 0:
        earliest_ts = pd.NA
        latest_ts = pd.NA
    else:
        earliest_ts = int(df["timestamp"].min())
        latest_ts = int(df["timestamp"].max())
    # unique cmdb_id values (sorted) limited to 20 entries
    if "cmdb_id" in df.columns:
        unique_cmdb = sorted(df["cmdb_id"].dropna().unique().tolist())
    else:
        unique_cmdb = []
    unique_cmdb_display = unique_cmdb[:20]
    summary = pd.DataFrame([{
        "file": name,
        "total_rows": total_rows,
        "earliest_ts": earliest_ts,
        "latest_ts": latest_ts,
        "unique_cmdb_count": len(unique_cmdb),
        "unique_cmdb_sample_up_to_20": unique_cmdb_display
    }])
    # if key_name provided, compute top up to 20 values with counts
    top_keys = pd.DataFrame()
    if key_name and key_name in df.columns:
        top_keys = (
            df[key_name]
            .fillna("N/A")
            .astype(str)
            .value_counts()
            .head(20)
            .rename_axis(key_name)
            .reset_index(name="count")
        )
    return summary, top_keys

# Build summaries and top lists
metric_summary, top_kpi_df = compact_summary(metric_df, "metric.csv", key_name="kpi_name")
trace_summary, top_trace_df = compact_summary(trace_df, "trace.csv", key_name="trace_name")
log_summary, top_log_df = compact_summary(log_df, "log.csv", key_name="log_name")

# For error_logs, top distinct message samples (treat empty/NaN as 'N/A')
# and limit to top 20
error_summary, _ = compact_summary(error_df, "error_logs.csv", key_name=None)
if "message" in error_df.columns:
    top_error_msgs_df = (
        error_df["message"]
        .fillna("")
        .astype(str)
        .apply(lambda x: x.strip() if x.strip() != "" else "N/A")
        .value_counts()
        .head(20)
        .rename_axis("message_sample")
        .reset_index(name="count")
    )
else:
    top_error_msgs_df = pd.DataFrame(columns=["message_sample", "count"])

# Final compact outputs (keep them concise)
metric_summary, trace_summary, log_summary, error_summary, top_kpi_df, top_trace_df, top_log_df, top_error_msgs_df

```
Out[1]:
```
```python
# Build a plain-English summary string from previously loaded summaries
metric_row = metric_summary.iloc[0]
trace_row = trace_summary.iloc[0]
log_row = log_summary.iloc[0]
error_row = error_summary.iloc[0]

def top_list_to_str(df, key_col):
    if df is None or df.shape[0] == 0:
        return "None"
    items = [f"{row[ key_col ]} ({int(row['count'])})" for _, row in df.iterrows()]
    return ", ".join(items)

metric_top = top_list_to_str(top_kpi_df, "kpi_name")
trace_top = top_list_to_str(top_trace_df, "trace_name")
log_top = top_list_to_str(top_log_df, "log_name")
error_top = top_list_to_str(top_error_msgs_df, "message_sample")

summary = (
    "Telemetry files overview:\n\n"
    f"metric.csv: {int(metric_row['total_rows'])} rows; earliest_ts={int(metric_row['earliest_ts'])}, "
    f"latest_ts={int(metric_row['latest_ts'])}; unique cmdb_ids ({int(metric_row['unique_cmdb_count'])}) "
    f"sample: {metric_row['unique_cmdb_sample_up_to_20']}\n"
    f"Top KPI names (up to 20, name (count)): {metric_top}\n\n"
    f"trace.csv: {int(trace_row['total_rows'])} rows; earliest_ts={int(trace_row['earliest_ts'])}, "
    f"latest_ts={int(trace_row['latest_ts'])}; unique cmdb_ids ({int(trace_row['unique_cmdb_count'])}) "
    f"sample: {trace_row['unique_cmdb_sample_up_to_20']}\n"
    f"Top trace names (up to 20, name (count)): {trace_top}\n\n"
    f"log.csv: {int(log_row['total_rows'])} rows; earliest_ts={int(log_row['earliest_ts'])}, "
    f"latest_ts={int(log_row['latest_ts'])}; unique cmdb_ids ({int(log_row['unique_cmdb_count'])}) "
    f"sample: {log_row['unique_cmdb_sample_up_to_20']}\n"
    f"Top log names (up to 20, name (count)): {log_top}\n\n"
    f"error_logs.csv: {int(error_row['total_rows'])} rows; earliest_ts={error_row['earliest_ts']}, "
    f"latest_ts={error_row['latest_ts']}; unique cmdb_ids ({int(error_row['unique_cmdb_count'])}) sample: "
    f"{error_row['unique_cmdb_sample_up_to_20']}\n"
    f"Top error message samples (up to 20): {error_top}\n\n"
    "Notes: error_logs.csv is empty. KPI/trace/log top lists show counts from the loaded data."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  earliest_ts   latest_ts  unique_cmdb_count                        unique_cmdb_sample_up_to_20
0  metric.csv       27691   1615012200  1615014000                 29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...,         file  total_rows  earliest_ts   latest_ts  unique_cmdb_count                        unique_cmdb_sample_up_to_20
0  trace.csv        6759   1615012200  1615013940                 12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...,       file  total_rows  earliest_ts   latest_ts  unique_cmdb_count                        unique_cmdb_sample_up_to_20
0  log.csv         463   1615012200  1615014000                  6  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...,              file  total_rows earliest_ts latest_ts  unique_cmdb_count unique_cmdb_sample_up_to_20
0  error_logs.csv           0        <NA>      <NA>                  0                          [],                                              kpi_name  count
0          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    364
1                  OSLinux-OSLinux_ZABBIX_Host_Uptime    364
2       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    364
3                             OSLinux-CPU_CPU_CPULoad    362
4                         OSLinux-CPU_CPU_CPUidleutil    362
5                         OSLinux-CPU_CPU_CPUUserTime    362
6                          OSLinux-CPU_CPU_CPUSysTime    362
7                              OSLinux-CPU_CPU_CPUWio    362
8   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    361
9                                             app.cnt    341
10                                             app.sr    341
11                                            app.mrt    341
12                                             app.rr    341
13     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    336
14       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    334
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    295
16       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    293
17                         OSLinux-CPU_CPU_CPUCpuUtil    284
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    279
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    276,                        trace_name  count
0        trace.self.duration_mean    356
1         trace.self.duration_p95    356
2            trace.self.row_count    356
3      trace.to_MG01.duration_p95    236
4         trace.to_MG01.row_count    236
5     trace.to_MG02.duration_mean    236
6      trace.to_MG02.duration_p95    236
7         trace.to_MG02.row_count    236
8     trace.to_MG01.duration_mean    236
9       trace.from_MG01.row_count    120
10   trace.from_MG02.duration_p95    120
11      trace.from_MG02.row_count    120
12   trace.from_MG01.duration_p95    120
13  trace.from_MG02.duration_mean    120
14  trace.from_MG01.duration_mean    120
15      trace.from_IG01.row_count    116
16      trace.from_IG02.row_count    116
17   trace.from_IG02.duration_p95    116
18  trace.from_IG01.duration_mean    116
19   trace.from_IG01.duration_p95    116,            log_name  count
0   log.error_count    182
1     log.row_count    182
2  log.gc_pause_max     99, Empty DataFrame
Columns: [message_sample, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import datetime

# Reuse already-loaded DataFrames: metric_df, trace_df, log_df (assumed in kernel)

# Ensure timestamp and value columns are numeric
for df in (metric_df, trace_df, log_df):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

# 1) Aggregate metric.csv by (cmdb_id, kpi_name) using the full series
agg_metric = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], dropna=False)
    .agg(
        count=("value", "count"),
        earliest_ts=("timestamp", "min"),
        latest_ts=("timestamp", "max"),
        p50=("value", lambda x: x.quantile(0.5) if x.notna().any() else pd.NA),
        p95=("value", lambda x: x.quantile(0.95) if x.notna().any() else pd.NA),
    )
    .reset_index()
)

# compact top 20 by count
agg_metric_top20 = agg_metric.sort_values("count", ascending=False).head(20)[
    ["cmdb_id", "kpi_name", "count", "earliest_ts", "latest_ts", "p50", "p95"]
]

# 2) Aggregate trace.csv by (cmdb_id, trace_name)
agg_trace = (
    trace_df
    .groupby(["cmdb_id", "trace_name"], dropna=False)
    .agg(
        count=("value", "count"),
        earliest_ts=("timestamp", "min"),
        latest_ts=("timestamp", "max"),
        p50=("value", lambda x: x.quantile(0.5) if x.notna().any() else pd.NA),
        p95=("value", lambda x: x.quantile(0.95) if x.notna().any() else pd.NA),
    )
    .reset_index()
)

agg_trace_top20 = agg_trace.sort_values("count", ascending=False).head(20)[
    ["cmdb_id", "trace_name", "count", "earliest_ts", "latest_ts", "p50", "p95"]
]

# 3) Aggregate log.csv by (cmdb_id, log_name)
agg_log = (
    log_df
    .groupby(["cmdb_id", "log_name"], dropna=False)
    .agg(
        count=("value", "count"),
        earliest_ts=("timestamp", "min"),
        latest_ts=("timestamp", "max"),
        p50=("value", lambda x: x.quantile(0.5) if x.notna().any() else pd.NA),
        p95=("value", lambda x: x.quantile(0.95) if x.notna().any() else pd.NA),
    )
    .reset_index()
)

agg_log_top20 = agg_log.sort_values("count", ascending=False).head(20)[
    ["cmdb_id", "log_name", "count", "earliest_ts", "latest_ts", "p50", "p95"]
]

# 4) Filter original dataframes to the incident time window in UTC+8:
tz = pytz.timezone("Asia/Shanghai")
start_dt = tz.localize(datetime.datetime(2021, 3, 6, 14, 30, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 6, 15, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

metric_window = metric_df[(metric_df["timestamp"] >= start_ts) & (metric_df["timestamp"] <= end_ts)].copy()
trace_window = trace_df[(trace_df["timestamp"] >= start_ts) & (trace_df["timestamp"] <= end_ts)].copy()
log_window = log_df[(log_df["timestamp"] >= start_ts) & (log_df["timestamp"] <= end_ts)].copy()

# Summaries for each filtered file
def window_summary(df, name):
    total_rows = int(len(df))
    if total_rows == 0:
        earliest = pd.NA
        latest = pd.NA
        unique_cmdb = []
    else:
        earliest = int(df["timestamp"].min())
        latest = int(df["timestamp"].max())
        unique_cmdb = sorted(df["cmdb_id"].dropna().unique().tolist())[:20]
    return pd.DataFrame([{
        "file": name,
        "total_rows_in_window": total_rows,
        "earliest_ts_in_window": earliest,
        "latest_ts_in_window": latest,
        "unique_cmdb_sample_up_to_20": unique_cmdb
    }])

metric_window_summary = window_summary(metric_window, "metric.csv")
trace_window_summary = window_summary(trace_window, "trace.csv")
log_window_summary = window_summary(log_window, "log.csv")

# For metric.csv: top 20 (cmdb_id, kpi_name) pairs by row count within the window
if not metric_window.empty:
    metric_window_top_pairs = (
        metric_window
        .groupby(["cmdb_id", "kpi_name"], dropna=False)
        .size()
        .reset_index(name="count_in_window")
        .sort_values("count_in_window", ascending=False)
        .head(20)
    )
else:
    metric_window_top_pairs = pd.DataFrame(columns=["cmdb_id", "kpi_name", "count_in_window"])

# Display compact results (top aggregated tables and window summaries)
agg_metric_top20, agg_trace_top20, agg_log_top20, metric_window_summary, metric_window_top_pairs, trace_window_summary, log_window_summary

```
Out[2]:
```
```python
# Build a plain-English summary string using existing kernel variables
# (agg_metric_top20, agg_trace_top20, agg_log_top20, metric_window_summary,
#  metric_window_top_pairs, trace_window_summary, log_window_summary,
#  agg_metric, agg_trace, agg_log, metric_window, trace_window, log_window)

# Helper to sample top N rows into brief strings
def sample_rows(df, cols, n=5):
    if df is None or df.shape[0] == 0:
        return "None"
    rows = []
    for _, r in df.head(n).iterrows():
        parts = [str(r[c]) for c in cols if c in r.index]
        rows.append(" / ".join(parts))
    return "; ".join(rows)

metric_examples = sample_rows(agg_metric_top20, ["cmdb_id", "kpi_name", "count"], n=6)
trace_examples = sample_rows(agg_trace_top20, ["cmdb_id", "trace_name", "count"], n=6)
log_examples = sample_rows(agg_log_top20, ["cmdb_id", "log_name", "count"], n=6)

mw = metric_window_summary.iloc[0]
tw = trace_window_summary.iloc[0]
lw = log_window_summary.iloc[0]

# Top metric pairs in window: sample few
metric_window_pairs_sample = sample_rows(metric_window_top_pairs, ["cmdb_id", "kpi_name", "count_in_window"], n=6)

summary = (
    "Summary of results (concise):\n\n"
    "1) Aggregations stored:\n"
    "- agg_metric, agg_trace, agg_log contain full aggregated tables by (cmdb_id, kpi_name/trace_name/log_name).\n\n"
    "2) metric.csv aggregation (top 20 by count):\n"
    f"- Top entries (examples: cmdb_id / kpi_name / count): {metric_examples}\n"
    "- Observation: many metric groups have count = 31 (regular per-minute series over the full time span).\n\n"
    "3) trace.csv aggregation (top 20 by count):\n"
    f"- Top entries (examples: cmdb_id / trace_name / count): {trace_examples}\n"
    "- Observation: prominent trace groups (e.g., trace.self.* and trace.to_* per IG01/IG02) have count = 30.\n\n"
    "4) log.csv aggregation (top 20 by count):\n"
    f"- Top entries (examples: cmdb_id / log_name / count): {log_examples}\n"
    "- Observation: common log metrics like log.error_count and log.row_count for Mysql01 and Tomcat nodes appear with count = 31; some gc_pause series have slightly fewer samples.\n\n"
    "5) Incident window filter (Mar 6, 2021 14:30:00 to 15:00:00 UTC+8):\n"
    f"- metric.csv in-window: total_rows_in_window = {int(mw['total_rows_in_window'])}, earliest_ts_in_window = {mw['earliest_ts_in_window']}, latest_ts_in_window = {mw['latest_ts_in_window']}\n"
    f"  Unique cmdb_id sample (up to 20): {mw['unique_cmdb_sample_up_to_20']}\n"
    f"  Top (cmdb_id, kpi_name) pairs by count in window (examples): {metric_window_pairs_sample}\n\n"
    f"- trace.csv in-window: total_rows_in_window = {int(tw['total_rows_in_window'])}, earliest_ts_in_window = {tw['earliest_ts_in_window']}, latest_ts_in_window = {tw['latest_ts_in_window']}\n"
    f"  Unique cmdb_id sample (up to 20): {tw['unique_cmdb_sample_up_to_20']}\n\n"
    f"- log.csv in-window: total_rows_in_window = {int(lw['total_rows_in_window'])}, earliest_ts_in_window = {lw['earliest_ts_in_window']}, latest_ts_in_window = {lw['latest_ts_in_window']}\n"
    f"  Unique cmdb_id sample (up to 20): {lw['unique_cmdb_sample_up_to_20']}\n\n"
    "Notes:\n"
    "- Full aggregated tables are stored as variables (agg_metric, agg_trace, agg_log).\n"
    "- Filtered per-file tables for the incident window are stored as metric_window, trace_window, log_window.\n"
    "- All numeric timestamps are in seconds (UTC+8 context used for filtering).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count  earliest_ts   latest_ts          p50          p95
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31   1615012200  1615014000         70.0         74.0
573    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31   1615012200  1615014000          0.0          0.0
574    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31   1615012200  1615014000          0.0          0.0
1503  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31   1615012200  1615014000          0.0          0.0
1400  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31   1615012200  1615014000          0.0          0.0
1401  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31   1615012200  1615014000          0.0          0.0
1402  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31   1615012200  1615014000          0.0          0.0
1403  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31   1615012200  1615014000        100.0        100.0
67        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31   1615012200  1615014000          0.0          0.0
68        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31   1615012200  1615014000          0.0          0.0
1498  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31   1615012200  1615014000         41.0         41.0
1499  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31   1615012200  1615014000     361406.0     361406.0
1500  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31   1615012200  1615014000  216546289.0  219847530.5
1501  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31   1615012200  1615014000     382073.0     388237.0
1502  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31   1615012200  1615014000          0.0          0.0
93        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31   1615012200  1615014000   57949328.0   58057952.0
81        IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31   1615012200  1615014000          0.3          0.3
82        IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31   1615012200  1615014000          3.0          3.0
78        IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31   1615012200  1615014000          1.0          1.0
1508  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31   1615012200  1615014000          0.0          0.0,    cmdb_id                       trace_name  count  earliest_ts   latest_ts          p50          p95
0     IG01         trace.self.duration_mean     30   1615012200  1615013940     0.667624     4.166548
1     IG01          trace.self.duration_p95     30   1615012200  1615013940     3.103500    15.561600
2     IG01             trace.self.row_count     30   1615012200  1615013940  1925.000000  2153.100000
3     IG01  trace.to_Tomcat01.duration_mean     30   1615012200  1615013940     0.617178     0.908307
4     IG01   trace.to_Tomcat01.duration_p95     30   1615012200  1615013940     2.266325     4.254033
5     IG01      trace.to_Tomcat01.row_count     30   1615012200  1615013940   240.000000   261.100000
6     IG01  trace.to_Tomcat02.duration_mean     30   1615012200  1615013940     0.604869     1.023005
7     IG01   trace.to_Tomcat02.duration_p95     30   1615012200  1615013940     2.226775     4.471105
8     IG01      trace.to_Tomcat02.row_count     30   1615012200  1615013940   236.500000   277.100000
13    IG01   trace.to_Tomcat04.duration_p95     30   1615012200  1615013940     2.240450     4.300455
12    IG01  trace.to_Tomcat04.duration_mean     30   1615012200  1615013940     0.587821     1.041199
14    IG01      trace.to_Tomcat04.row_count     30   1615012200  1615013940   242.500000   285.350000
17    IG02             trace.self.row_count     30   1615012200  1615013940  1933.000000  2247.100000
16    IG02          trace.self.duration_p95     30   1615012200  1615013940     2.862700    15.560312
15    IG02         trace.self.duration_mean     30   1615012200  1615013940     0.682674     4.336802
23    IG02      trace.to_Tomcat02.row_count     30   1615012200  1615013940   245.000000   288.150000
29    IG02      trace.to_Tomcat04.row_count     30   1615012200  1615013940   248.500000   291.800000
28    IG02   trace.to_Tomcat04.duration_p95     30   1615012200  1615013940     2.176300     4.167770
18    IG02  trace.to_Tomcat01.duration_mean     30   1615012200  1615013940     0.540183     0.879065
19    IG02   trace.to_Tomcat01.duration_p95     30   1615012200  1615013940     2.271525     3.771672,      cmdb_id          log_name  count  earliest_ts   latest_ts      p50        p95
0    Mysql01   log.error_count     31   1615012200  1615014000    0.000     0.0000
1    Mysql01     log.row_count     31   1615012200  1615014000    8.000    21.0000
2   Tomcat01   log.error_count     31   1615012200  1615014000    0.000     0.0000
4   Tomcat01     log.row_count     31   1615012200  1615014000  474.000   541.5000
7   Tomcat02     log.row_count     31   1615012200  1615014000  478.000   555.0000
5   Tomcat02   log.error_count     31   1615012200  1615014000    0.000     0.0000
15  apache02     log.row_count     31   1615012200  1615014000  961.000  1074.0000
11  Tomcat04   log.error_count     31   1615012200  1615014000    0.000     0.0000
14  apache02   log.error_count     31   1615012200  1615014000    0.000     0.0000
13  Tomcat04     log.row_count     31   1615012200  1615014000  487.000   556.5000
6   Tomcat02  log.gc_pause_max     30   1615012200  1615013940    0.070     0.1155
8   Tomcat03   log.error_count     27   1615012200  1615014000    0.000     0.0000
10  Tomcat03     log.row_count     27   1615012200  1615014000  495.000   542.9000
3   Tomcat01  log.gc_pause_max     24   1615012200  1615013940    0.065     0.0900
12  Tomcat04  log.gc_pause_max     24   1615012260  1615013940    0.070     0.1000
9   Tomcat03  log.gc_pause_max     21   1615012200  1615013940    0.070     0.0800,          file  total_rows_in_window  earliest_ts_in_window  latest_ts_in_window                        unique_cmdb_sample_up_to_20
0  metric.csv                 27691             1615012200           1615014000  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...,        cmdb_id                                           kpi_name  count_in_window
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads               31
573    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31
574    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31
1503  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...               31
1400  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               31
1401  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               31
1402  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               31
1403  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31
67        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31
68        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31
1498  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...               31
1499  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...               31
1500  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...               31
1501  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...               31
1502  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter               31
93        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31
81        IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31
82        IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31
78        IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31
1508  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...               31,         file  total_rows_in_window  earliest_ts_in_window  latest_ts_in_window                        unique_cmdb_sample_up_to_20
0  trace.csv                  6759             1615012200           1615013940  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...,       file  total_rows_in_window  earliest_ts_in_window  latest_ts_in_window                        unique_cmdb_sample_up_to_20
0  log.csv                   463             1615012200           1615014000  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing kernel variables: agg_metric, metric_window, agg_trace, trace_window, agg_log, log_window

# Ensure numeric types
for df in (metric_window, trace_window, log_window):
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

# --- Helper to perform anomaly scan for a generic dataset ---
def anomaly_scan(agg_df, window_df, key_col, name_col):
    # agg_df: aggregated table containing p95 column
    # window_df: filtered window rows containing value and timestamp
    # key_col: "cmdb_id"
    # name_col: the KPI/trace/log name column in both tables (e.g., "kpi_name")
    p95_col = "p95"
    # Prepare list of groups with missing p95
    agg_p95 = agg_df[[key_col, name_col, p95_col]].copy()
    # Identify groups with missing/NaN p95
    missing_mask = agg_p95[p95_col].isna()
    missing_groups = (
        agg_p95.loc[missing_mask, [key_col, name_col]]
        .astype(str)
        .apply(lambda row: f"{row[key_col]}|{row[name_col]}", axis=1)
        .tolist()
    )
    missing_groups_sample = missing_groups[:20]
    # Keep only groups with valid p95
    valid_p95 = agg_p95.loc[~missing_mask].copy()
    # Merge p95 into window rows by key and name
    merged = pd.merge(
        window_df,
        valid_p95,
        how="left",
        on=[key_col, name_col],
        suffixes=("", "_agg")
    )
    # Drop rows where p95 is NA (these window rows don't have agg group or had missing p95)
    merged_valid = merged[merged[p95_col].notna()].copy()
    if merged_valid.empty:
        # No candidate anomalies
        empty_cols = [key_col, name_col, "anomaly_count", "earliest_anomaly_ts", "latest_anomaly_ts",
                      "max_value_in_window", "p95_threshold", "severity_ratio"]
        return pd.DataFrame(columns=empty_cols), missing_groups_sample

    # Filter rows where value >= p95
    # Note: p95 is global threshold (from agg_df) per group
    cond = merged_valid["value"] >= merged_valid[p95_col]
    anomalies_rows = merged_valid[cond].copy()
    if anomalies_rows.empty:
        empty_cols = [key_col, name_col, "anomaly_count", "earliest_anomaly_ts", "latest_anomaly_ts",
                      "max_value_in_window", "p95_threshold", "severity_ratio"]
        return pd.DataFrame(columns=empty_cols), missing_groups_sample

    # Aggregate per group
    grouped = (
        anomalies_rows
        .groupby([key_col, name_col], dropna=False)
        .agg(
            anomaly_count=("value", "count"),
            earliest_anomaly_ts=("timestamp", "min"),
            latest_anomaly_ts=("timestamp", "max"),
            max_value_in_window=("value", "max"),
            p95_threshold=(p95_col, "first")
        )
        .reset_index()
    )

    # Compute severity_ratio as specified
    def compute_severity(row):
        thr = row["p95_threshold"]
        mv = row["max_value_in_window"]
        try:
            if pd.isna(thr):
                return np.nan
            if thr > 0:
                return mv / thr
            else:
                return mv
        except Exception:
            return np.nan

    grouped["severity_ratio"] = grouped.apply(compute_severity, axis=1)

    # Keep only groups with anomaly_count > 0 (they already are) and drop any with missing p95_threshold just in case
    grouped = grouped[grouped["p95_threshold"].notna()].copy()

    # Sort by anomaly_count desc then severity_ratio desc
    grouped_sorted = grouped.sort_values(["anomaly_count", "severity_ratio"], ascending=[False, False]).reset_index(drop=True)

    # Return full table and missing samples
    return grouped_sorted, missing_groups_sample

# --- 1) Metric anomaly scan ---
metric_anoms, missing_metric = anomaly_scan(agg_metric, metric_window, "cmdb_id", "kpi_name")
# Store full anomalies table in kernel variable metric_anoms (already assigned)
# Prepare top 20 compact view
metric_anoms_top20 = metric_anoms[["cmdb_id", "kpi_name", "anomaly_count", "earliest_anomaly_ts",
                                   "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]].head(20)

# --- 2) Trace anomaly scan ---
trace_anoms, missing_trace = anomaly_scan(agg_trace, trace_window, "cmdb_id", "trace_name")
trace_anoms_top20 = trace_anoms[["cmdb_id", "trace_name", "anomaly_count", "earliest_anomaly_ts",
                                 "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]].head(20)

# --- 3) Log anomaly scan ---
log_anoms, missing_log = anomaly_scan(agg_log, log_window, "cmdb_id", "log_name")
log_anoms_top20 = log_anoms[["cmdb_id", "log_name", "anomaly_count", "earliest_anomaly_ts",
                             "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]].head(20)

# Limit missing lists to up to 20 names each (already done)
missing_metric_sample = missing_metric[:20]
missing_trace_sample = missing_trace[:20]
missing_log_sample = missing_log[:20]

# Final compact outputs (do not print long series)
metric_anoms_top20, trace_anoms_top20, log_anoms_top20, missing_metric_sample, missing_trace_sample, missing_log_sample

```
Out[3]:
```
Summary of anomaly scan results (incident window Mar 6, 2021 14:30–15:00 UTC+8):

General
- The full anomaly tables are stored in kernel variables: metric_anoms, trace_anoms, log_anoms.
- No aggregation groups were skipped due to missing p95 (missing-groups lists are empty).

Metric anomalies (top results)
- Many metric groups triggered anomalies across the entire window (anomaly_count = 31 for top groups, meaning every minute sample in the window ≥ the group's global p95).
- Top examples (cmdb_id / kpi_name / anomaly_count / severity_ratio):
  - Mysql02 / OSLinux-..._zabbix_agentd_PROC... / 31 / 1.2
  - IG01 / JVM-Memory_..._HeapMemoryMax / 31 / 1.0
  - IG01 / OSLinux..._PROCESS_PROCNoZombies / 31 / 1.0
  - IG01 / OSLinux..._zabbix_PROCPPCount / 31 / 1.0
  - Tomcat01 / various Tomcat request/thread KPIs / 31 / 1.0
- Interpretation: Many metric groups were at-or-above their global 95th-percentile through the whole window; the highest severity_ratio shown is ~1.2 for a Mysql02 zabbix/process metric.

Trace anomalies (top results)
- Several trace series exceeded their global p95, with varying counts and severities.
- Top examples (cmdb_id / trace_name / anomaly_count / severity_ratio):
  - dockerB2 / trace.self.duration_p95 / 5 / 1.000989
  - dockerB1 / trace.self.duration_p95 / 5 / 1.000984
  - Tomcat03 / trace.self.duration_p95 / 3 / 1.014691
  - MG01 / trace.from_dockerA2.duration_p95 / 2 / 2.158358 (notably high severity)
  - dockerA2 / trace.to_MG02.duration_p95 / 2 / 1.946967
- Interpretation: Some trace paths (especially flows involving dockerA*/MG* components) show elevated durations relative to their global p95; a few have severity ratios >1.9, indicating values substantially above the p95.

Log anomalies (top results)
- Several log.error_count series have anomaly_count = 31 but their values and p95 thresholds are zero, yielding severity_ratio = 0 (i.e., values = 0 and threshold = 0).
- Notable non-zero log anomalies:
  - Tomcat04 / log.gc_pause_max — anomaly_count 3, severity ~1.20
  - Tomcat01 / log.gc_pause_max — anomaly_count 3, severity ~1.11
  - Tomcat03 / log.gc_pause_max — anomaly_count 4, severity = 1.00
  - Some log.row_count entries (Mysql01, Tomcat03, Tomcat02, apache02) show minor elevations (severity ~1.03–1.10).
- Interpretation: GC pause metrics on some Tomcat nodes exceeded their p95 by ~10–20%; several row_count metrics modestly exceeded p95.

Key takeaways
- Multiple metric groups across IG*, MG*, Tomcat*, Mysql*, Redis* have sustained values ≥ their global p95 throughout the incident window—this is broad and may indicate system-wide load or sampling of high-percentile behavior.
- Trace anomalies highlight particular flows (dockerA*/dockerB* ↔ MG*) with the largest severity ratios; these are potential hotspots for increased latency.
- GC pause spikes on Tomcat nodes are present and could contribute to service latency; log.error_count metrics remain at zero in many cases (so errors did not increase).
- Full anomaly tables (metric_anoms, trace_anoms, log_anoms) contain all detected groups and can be inspected for details per group and timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31           1615012200         1615014000         6.000000e-01   5.000000e-01             1.2
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31           1615012200         1615014000         2.040136e+09   2.040136e+09             1.0
2       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31           1615012200         1615014000         1.000000e+00   1.000000e+00             1.0
3       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615012200         1615014000         3.000000e+00   3.000000e+00             1.0
4       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31           1615012200         1615014000         2.040136e+09   2.040136e+09             1.0
5       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31           1615012200         1615014000         3.000000e-01   3.000000e-01             1.0
6       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31           1615012200         1615014000         2.040136e+09   2.040136e+09             1.0
7       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31           1615012200         1615014000         5.068464e+08   5.068464e+08             1.0
8       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31           1615012200         1615014000         2.040136e+09   2.040136e+09             1.0
9       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31           1615012200         1615014000         9.472855e+08   9.472855e+08             1.0
10   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615012200         1615014000         7.000000e+00   7.000000e+00             1.0
11   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615012200         1615014000         1.200000e+01   1.200000e+01             1.0
12   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31           1615012200         1615014000         1.000000e+00   1.000000e+00             1.0
13   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615012200         1615014000         4.000000e+00   4.000000e+00             1.0
14   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615012200         1615014000         4.000000e+00   4.000000e+00             1.0
15  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31           1615012200         1615014000         2.300000e+01   2.300000e+01             1.0
16  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31           1615012200         1615014000         3.488470e+05   3.488470e+05             1.0
17  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31           1615012200         1615014000         1.000000e+02   1.000000e+02             1.0
18  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31           1615012200         1615014000         5.000000e+02   5.000000e+02             1.0
19  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31           1615012200         1615014000         4.000000e+01   4.000000e+01             1.0,      cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0   dockerB2            trace.self.duration_p95              5           1615012320         1615013880             1.012000       1.011000        1.000989
1   dockerB1            trace.self.duration_p95              5           1615012320         1615013100             1.017000       1.016000        1.000984
2   Tomcat03            trace.self.duration_p95              3           1615012380         1615013880             1.036000       1.021000        1.014691
3       MG01      trace.from_Tomcat03.row_count              3           1615012200         1615012920           281.000000     277.000000        1.014440
4   Tomcat03            trace.to_MG01.row_count              3           1615012200         1615012920           281.000000     277.000000        1.014440
5       MG01   trace.from_dockerA2.duration_p95              2           1615012800         1615013040             2.472350       1.145477        2.158358
6   dockerA2         trace.to_MG01.duration_p95              2           1615012800         1615013040             2.472350       1.145477        2.158358
7       MG02  trace.from_dockerA2.duration_mean              2           1615012800         1615013040             0.318225       0.163286        1.948880
8   dockerA2        trace.to_MG02.duration_mean              2           1615012800         1615013040             0.318225       0.163286        1.948880
9       MG02   trace.from_dockerA2.duration_p95              2           1615012800         1615013040             2.069650       1.063012        1.946967
10  dockerA2         trace.to_MG02.duration_p95              2           1615012800         1615013040             2.069650       1.063012        1.946967
11      MG02   trace.from_dockerB1.duration_p95              2           1615012680         1615013040             1.778200       1.035355        1.717479
12  dockerB1         trace.to_MG02.duration_p95              2           1615012680         1615013040             1.778200       1.035355        1.717479
13      MG01   trace.from_dockerA1.duration_p95              2           1615012800         1615013040             1.859250       1.152152        1.613719
14  dockerA1         trace.to_MG01.duration_p95              2           1615012800         1615013040             1.859250       1.152152        1.613719
15      MG02  trace.from_dockerA1.duration_mean              2           1615012680         1615013040             0.216432       0.145576        1.486723
16  dockerA1        trace.to_MG02.duration_mean              2           1615012680         1615013040             0.216432       0.145576        1.486723
17      MG01  trace.from_dockerA1.duration_mean              2           1615012800         1615013040             0.309462       0.209551        1.476783
18  dockerA1        trace.to_MG01.duration_mean              2           1615012800         1615013040             0.309462       0.209551        1.476783
19      MG02      trace.from_dockerA1.row_count              2           1615012200         1615012920           104.000000      79.550000        1.307354,      cmdb_id          log_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0    Mysql01   log.error_count             31           1615012200         1615014000                 0.00         0.0000        0.000000
1   Tomcat01   log.error_count             31           1615012200         1615014000                 0.00         0.0000        0.000000
2   Tomcat02   log.error_count             31           1615012200         1615014000                 0.00         0.0000        0.000000
3   Tomcat04   log.error_count             31           1615012200         1615014000                 0.00         0.0000        0.000000
4   apache02   log.error_count             31           1615012200         1615014000                 0.00         0.0000        0.000000
5   Tomcat03   log.error_count             27           1615012200         1615014000                 0.00         0.0000        0.000000
6   Tomcat03  log.gc_pause_max              4           1615012320         1615013760                 0.08         0.0800        1.000000
7   Tomcat04  log.gc_pause_max              3           1615012380         1615013580                 0.12         0.1000        1.200000
8   Tomcat01  log.gc_pause_max              3           1615012920         1615013820                 0.10         0.0900        1.111111
9    Mysql01     log.row_count              2           1615012620         1615012800                23.00        21.0000        1.095238
10  Tomcat03     log.row_count              2           1615012380         1615012920               594.00       542.9000        1.094124
11  Tomcat02  log.gc_pause_max              2           1615012680         1615013460                 0.12         0.1155        1.038961
12  Tomcat02     log.row_count              2           1615012980         1615013880               573.00       555.0000        1.032432
13  apache02     log.row_count              2           1615012920         1615013880              1108.00      1074.0000        1.031657
14  Tomcat01     log.row_count              2           1615012440         1615013880               549.00       541.5000        1.013850
15  Tomcat04     log.row_count              2           1615012380         1615012920               562.00       556.5000        1.009883, [], [], [])```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse timezone object per rule
tz = pytz.timezone('Asia/Shanghai')

# Assume metric_anoms, trace_anoms, log_anoms exist in kernel from previous steps

# 1) Metric-driven component summary
# Ensure we operate on a DataFrame copy
ma = metric_anoms.copy()

# Only consider groups with anomaly_count > 0 (should already be the case)
ma = ma[ma["anomaly_count"] > 0].copy()

# Aggregate by cmdb_id
component_metric_summary = (
    ma
    .groupby("cmdb_id", dropna=False)
    .agg(
        num_anom_groups=("kpi_name", "nunique"),
        num_high_sev_groups=("severity_ratio", lambda s: int((s >= 1.3).sum())),
        max_severity_ratio=("severity_ratio", "max"),
        earliest_anomaly_ts=("earliest_anomaly_ts", "min")
    )
    .reset_index()
)

# Sort and prepare top 20 view
component_metric_summary = component_metric_summary.sort_values(
    ["num_anom_groups", "max_severity_ratio"], ascending=[False, False]
).reset_index(drop=True)

component_metric_summary_top20 = component_metric_summary.head(20)[
    ["cmdb_id", "num_anom_groups", "num_high_sev_groups", "max_severity_ratio", "earliest_anomaly_ts"]
]

# 2) Trace-driven high-severity anomalies
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

ta = trace_anoms.copy()
# Filter allowed components and severity >= 1.3
trace_candidate_anoms = ta[
    (ta["cmdb_id"].isin(allowed_components)) &
    (ta["severity_ratio"] >= 1.3)
].copy()

trace_candidate_anoms = trace_candidate_anoms.sort_values(
    ["severity_ratio", "anomaly_count"], ascending=[False, False]
).reset_index(drop=True)

trace_candidate_anoms_top20 = trace_candidate_anoms.head(20)[
    ["cmdb_id", "trace_name", "anomaly_count", "earliest_anomaly_ts",
     "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]
]

# 3) Log-driven anomalies of interest
la = log_anoms.copy()
log_candidate_anoms = la[
    (la["cmdb_id"].isin(allowed_components)) &
    (la["severity_ratio"] >= 1.05)
].copy()

log_candidate_anoms = log_candidate_anoms.sort_values(
    ["severity_ratio", "anomaly_count"], ascending=[False, False]
).reset_index(drop=True)

log_candidate_anoms_top20 = log_candidate_anoms.head(20)[
    ["cmdb_id", "log_name", "anomaly_count", "earliest_anomaly_ts",
     "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]
]

# Display compact results (DataFrame variables; per rule use variable names)
component_metric_summary_top20, trace_candidate_anoms_top20, log_candidate_anoms_top20

```
Out[4]:
```
```python
# Plain-English concise summary based on component_metric_summary, trace_candidate_anoms, log_candidate_anoms
summary = (
    "Fault-candidate summary (concise):\n\n"
    "1) Metric-driven candidates:\n"
    "- Mysql02 and Mysql01 show the most metric anomalies (226 anomalous KPI groups each). "
    "They also contain the largest number of high-severity groups (Mysql02:16, Mysql01:4).\n"
    "- Several Tomcat nodes (Tomcat01–Tomcat04) and Redis nodes also show many anomalous metric groups "
    "and some high-severity ratios (notably Tomcat01–04, Redis02).\n"
    "- MG01/MG02 and IG01/IG02 appear with substantial counts and elevated max severity ratios as well.\n\n"
    "2) Trace-driven high-severity candidates (allowed components):\n"
    "- MG01 and MG02 surface as top trace suspects. High-severity trace anomalies involve paths from dockerA*/dockerB* "
    "to MG nodes (e.g., trace.from_dockerA2.duration_p95 with severity ~2.16 on MG01; similar patterns on MG02). "
    "These indicate notable latency spikes on flows involving dockerA*/dockerB* ↔ MG*.\n\n"
    "3) Log-driven candidates (allowed components):\n"
    "- GC pause spikes observed on Tomcat nodes (Tomcat04 severity ~1.20; Tomcat01 ~1.11). "
    "These are plausible contributors to latency.\n"
    "- Some log.row_count increases (Mysql01, Tomcat03, Tomcat02, apache02) show modest severity (~1.03–1.10).\n"
    "- log.error_count remains at zero for many components (no broad increase in logged errors).\n\n"
    "Overall recommendations (prioritized):\n"
    "1. Investigate Mysql01/Mysql02 metric anomalies and high-severity groups for resource pressure or long-running processes.\n"
    "2. Inspect Tomcat cluster GC behavior (Tomcat04, Tomcat01) and corresponding request handling spikes.\n"
    "3. Examine network/trace paths between dockerA*/dockerB* and MG01/MG02 for latency hotspots (trace anomalies with high severity).\n"
    "4. Review Redis and IG/MG components next, as they show elevated metric counts and some high severity ratios.\n\n"
    "Stored variables for further drill-down: component_metric_summary, trace_candidate_anoms, log_candidate_anoms."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  num_anom_groups  num_high_sev_groups  max_severity_ratio  earliest_anomaly_ts
0        Mysql02              226                   16            4.000000           1615012200
1        Mysql01              226                    4            3.297751           1615012200
2        Redis01              113                    2            1.797022           1615012200
3       Tomcat04              111                    6            1.714914           1615012200
4       Tomcat02              109                    6           10.000000           1615012200
5       Tomcat03              109                    5           10.000000           1615012200
6        Redis02              109                    2            3.333333           1615012200
7       Tomcat01              108                    6           20.000000           1615012200
8           MG02               92                    5            1.887446           1615012200
9           MG01               91                    4            6.666667           1615012200
10          IG01               90                    4            3.333333           1615012200
11          IG02               89                    4            4.000000           1615012200
12      apache02               81                    4            3.333333           1615012200
13      apache01               37                   10            1.923552           1615012200
14      dockerA2               21                    0            1.036951           1615012200
15      dockerB1                6                    0            1.078116           1615012560
16      dockerB2                6                    0            1.019581           1615012500
17      dockerA1                5                    0            1.031013           1615012620
18  ServiceTest3                4                    1            1.438646           1615012200
19  ServiceTest9                4                    1            1.392662           1615012200,     cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0      MG01   trace.from_dockerA2.duration_p95              2           1615012800         1615013040             2.472350       1.145477        2.158358
1      MG02  trace.from_dockerA2.duration_mean              2           1615012800         1615013040             0.318225       0.163286        1.948880
2      MG02   trace.from_dockerA2.duration_p95              2           1615012800         1615013040             2.069650       1.063012        1.946967
3      MG02   trace.from_dockerB1.duration_p95              2           1615012680         1615013040             1.778200       1.035355        1.717479
4      MG01   trace.from_dockerA1.duration_p95              2           1615012800         1615013040             1.859250       1.152152        1.613719
5      MG02  trace.from_dockerA1.duration_mean              2           1615012680         1615013040             0.216432       0.145576        1.486723
6      MG01  trace.from_dockerA1.duration_mean              2           1615012800         1615013040             0.309462       0.209551        1.476783
7      MG02      trace.from_dockerA1.row_count              2           1615012200         1615012920           104.000000      79.550000        1.307354
8      MG02  trace.from_Tomcat02.duration_mean              2           1615012380         1615012920             0.938352       0.720786        1.301846
9  Tomcat02        trace.to_MG02.duration_mean              2           1615012380         1615012920             0.938352       0.720786        1.301846,     cmdb_id          log_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0  Tomcat04  log.gc_pause_max              3           1615012380         1615013580                 0.12           0.10        1.200000
1  Tomcat01  log.gc_pause_max              3           1615012920         1615013820                 0.10           0.09        1.111111
2   Mysql01     log.row_count              2           1615012620         1615012800                23.00          21.00        1.095238
3  Tomcat03     log.row_count              2           1615012380         1615012920               594.00         542.90        1.094124)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing trace_window from kernel
# Filter for MG01 or MG02 and trace_name containing dockerA, dockerB, or Tomcat
pattern = r"(dockerA|dockerB|Tomcat)"
mask = trace_window["cmdb_id"].isin(["MG01", "MG02"]) & trace_window["trace_name"].str.contains(pattern, na=False)

mg_trace_rows_full = trace_window[mask].copy()
mg_trace_rows_full = mg_trace_rows_full.sort_values("timestamp", ascending=True)

# Compact display rows (limit to 20)
mg_trace_rows = mg_trace_rows_full[["timestamp", "cmdb_id", "trace_name", "value"]].head(20)

# Per-(cmdb_id, trace_name) summary computed from the full filtered set
mg_trace_summary_full = (
    mg_trace_rows_full
    .groupby(["cmdb_id", "trace_name"], dropna=False)
    .agg(
        anomaly_count_in_window=("value", "count"),
        earliest_ts_in_window=("timestamp", "min"),
        latest_ts_in_window=("timestamp", "max"),
        max_value_in_window=("value", "max")
    )
    .reset_index()
    .sort_values(["anomaly_count_in_window", "max_value_in_window"], ascending=[False, False])
)

# Compact summary (limit to 20)
mg_trace_summary = mg_trace_summary_full.head(20)[
    ["cmdb_id", "trace_name", "anomaly_count_in_window", "earliest_ts_in_window", "latest_ts_in_window", "max_value_in_window"]
]

# Return compact outputs (full tables kept in *_full variables for later use)
mg_trace_rows, mg_trace_summary

```
Out[5]:
```
```python
# Build a plain-English summary from existing kernel variables mg_trace_rows and mg_trace_summary
# (these were created from trace_window filtering for MG01/MG02 and dockerA/dockerB/Tomcat)

top_rows = mg_trace_rows.shape[0] if 'mg_trace_rows' in globals() else 0
group_count = mg_trace_summary.shape[0] if 'mg_trace_summary' in globals() else 0

# Sample up to 5 groups for examples
examples = []
if group_count > 0:
    for _, r in mg_trace_summary.head(5).iterrows():
        examples.append(f"{r['cmdb_id']} / {r['trace_name']} (count={int(r['anomaly_count_in_window'])}, max={r['max_value_in_window']})")

max_value_overall = None
if group_count > 0:
    max_value_overall = mg_trace_summary["max_value_in_window"].max()

summary = (
    "MG01/MG02 trace-window findings (concise):\n\n"
    f"- Rows returned (sampled up to 20): {top_rows} rows (sorted by timestamp ascending).\n"
    f"- Distinct (cmdb_id, trace_name) groups in summary: {group_count} (showing up to 20).\n"
    "- Key observations:\n"
    "  1) Most matching trace groups are steady across the incident window (many groups have ~30 samples, "
    "covering the interval 1615012200 → 1615013940), indicating continuous traffic during the window.\n"
    "  2) The groups are primarily row_count series (high counts) and duration metrics from Tomcat and dockerA/dockerB flows.\n"
    f"  3) Example groups: {', '.join(examples)}.\n"
    f"  4) Highest observed max value among these groups (in the summary) is {max_value_overall}.\n\n"
    "Interpretation / next steps:\n"
    "- The consistency (≈30 samples) suggests these flows were active throughout the incident window.\n"
    "- Inspect the top row_count flows (largest max values) and the duration_p95 series for dockerA/dockerB ↔ MG paths "
    "for latency or load impact. Prioritize groups with the largest max_value_in_window.\n\n"
    "Stored variables for drill-down: mg_trace_rows (sampled rows) and mg_trace_summary (per-group summary)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp cmdb_id                         trace_name       value
30  1615012200    MG01  trace.from_Tomcat01.duration_mean    0.504013
31  1615012200    MG01   trace.from_Tomcat01.duration_p95    2.236400
32  1615012200    MG01      trace.from_Tomcat01.row_count  237.000000
33  1615012200    MG01  trace.from_Tomcat02.duration_mean    0.415188
34  1615012200    MG01   trace.from_Tomcat02.duration_p95    2.078000
35  1615012200    MG01      trace.from_Tomcat02.row_count  261.000000
36  1615012200    MG01  trace.from_Tomcat03.duration_mean    0.496964
37  1615012200    MG01   trace.from_Tomcat03.duration_p95    2.107800
39  1615012200    MG01  trace.from_Tomcat04.duration_mean    0.448079
38  1615012200    MG01      trace.from_Tomcat03.row_count  277.000000
50  1615012200    MG01      trace.from_dockerB1.row_count  228.000000
51  1615012200    MG01  trace.from_dockerB2.duration_mean    0.031296
52  1615012200    MG01   trace.from_dockerB2.duration_p95    0.022000
53  1615012200    MG01      trace.from_dockerB2.row_count  169.000000
92  1615012200    MG02      trace.from_dockerB1.row_count  185.000000
93  1615012200    MG02  trace.from_dockerB2.duration_mean    0.063403
40  1615012200    MG01   trace.from_Tomcat04.duration_p95    2.079700
41  1615012200    MG01      trace.from_Tomcat04.row_count  254.000000
42  1615012200    MG01  trace.from_dockerA1.duration_mean    0.051597
43  1615012200    MG01   trace.from_dockerA1.duration_p95    0.041200,    cmdb_id                     trace_name  anomaly_count_in_window  earliest_ts_in_window  latest_ts_in_window  max_value_in_window
56    MG02  trace.from_dockerB1.row_count                       30             1615012200           1615013940                349.0
71    MG02    trace.to_dockerB2.row_count                       30             1615012200           1615013940                337.0
20    MG01  trace.from_dockerB1.row_count                       30             1615012200           1615013940                332.0
35    MG01    trace.to_dockerB2.row_count                       30             1615012200           1615013940                327.0
23    MG01  trace.from_dockerB2.row_count                       30             1615012200           1615013940                321.0
68    MG02    trace.to_dockerB1.row_count                       30             1615012200           1615013940                314.0
47    MG02  trace.from_Tomcat04.row_count                       30             1615012200           1615013940                305.0
32    MG01    trace.to_dockerB1.row_count                       30             1615012200           1615013940                303.0
59    MG02  trace.from_dockerB2.row_count                       30             1615012200           1615013940                303.0
5     MG01  trace.from_Tomcat02.row_count                       30             1615012200           1615013940                293.0
38    MG02  trace.from_Tomcat01.row_count                       30             1615012200           1615013940                293.0
41    MG02  trace.from_Tomcat02.row_count                       30             1615012200           1615013940                284.0
11    MG01  trace.from_Tomcat04.row_count                       30             1615012200           1615013940                283.0
2     MG01  trace.from_Tomcat01.row_count                       30             1615012200           1615013940                276.0
65    MG02    trace.to_dockerA2.row_count                       30             1615012200           1615013940                267.0
26    MG01    trace.to_dockerA1.row_count                       30             1615012200           1615013940                254.0
62    MG02    trace.to_dockerA1.row_count                       30             1615012200           1615013940                234.0
29    MG01    trace.to_dockerA2.row_count                       30             1615012200           1615013940                229.0
50    MG02  trace.from_dockerA1.row_count                       30             1615012200           1615013940                104.0
53    MG02  trace.from_dockerA2.row_count                       30             1615012200           1615013940                100.0)```
```

In [ ]:
# In[6]:

# Reuse existing kernel DataFrames: trace_candidate_anoms, metric_anoms
import pandas as pd

# 1) Earliest trace anomalies (up to 5 rows for the minimum earliest_anomaly_ts)
if "trace_candidate_anoms" in globals() and not trace_candidate_anoms.empty:
    min_ts = trace_candidate_anoms["earliest_anomaly_ts"].min()
    earliest_trace_anoms = (
        trace_candidate_anoms[trace_candidate_anoms["earliest_anomaly_ts"] == min_ts]
        .sort_values("earliest_anomaly_ts", ascending=True)
        .loc[:, ["cmdb_id", "trace_name", "anomaly_count", "earliest_anomaly_ts",
                 "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]]
        .head(5)
        .reset_index(drop=True)
    )
else:
    earliest_trace_anoms = pd.DataFrame(columns=[
        "cmdb_id", "trace_name", "anomaly_count", "earliest_anomaly_ts",
        "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"
    ])

# 2) MG01/MG02 metric anomalies with severity >= 1.3 and anomaly_count > 0 (top 20)
if "metric_anoms" in globals() and not metric_anoms.empty:
    mg_metric_highsev = (
        metric_anoms[
            metric_anoms["cmdb_id"].isin(["MG01", "MG02"]) &
            (metric_anoms["severity_ratio"] >= 1.3) &
            (metric_anoms["anomaly_count"] > 0)
        ]
        .loc[:, ["cmdb_id", "kpi_name", "anomaly_count", "earliest_anomaly_ts",
                 "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"]]
        .sort_values(["severity_ratio", "anomaly_count"], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )
else:
    mg_metric_highsev = pd.DataFrame(columns=[
        "cmdb_id", "kpi_name", "anomaly_count", "earliest_anomaly_ts",
        "latest_anomaly_ts", "max_value_in_window", "p95_threshold", "severity_ratio"
    ])

# Return the compact result tables (stored in kernel variables for later use)
earliest_trace_anoms, mg_metric_highsev

```
Out[6]:
```
# Plain-English concise summary based on earliest_trace_anoms and mg_metric_highsev (already in kernel)
summary = (
    "Findings:\n\n"
    "1) Earliest trace anomaly(s):\n"
    "- Earliest anomaly is on MG02: trace.from_dockerA1.row_count\n"
    "  • anomaly_count = 2\n"
    "  • earliest_anomaly_ts = 1615012200, latest_anomaly_ts = 1615012920\n"
    "  • max_value_in_window = 104.0, p95_threshold = 79.55, severity_ratio ≈ 1.31\n"
    "  → Interpretation: increased row_count from dockerA1 to MG02 started at the earliest timestamp in the candidate traces.\n\n"
    "2) MG01 / MG02 metric high-severity anomalies (severity >= 1.3):\n"
    "- MG01: OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT\n"
    "  • anomaly_count = 1, earliest = 1615013880, max = 1.0, p95 = 0.15, severity ≈ 6.67\n"
    "- MG02: several local-disk and I/O related metrics (examples):\n"
    "  • OSLinux-OSLinux_LOCALDISK...sda_DSKRead (anomaly_count=1, earliest=1615013040, max=852.4, p95=486.64, severity≈1.75)\n"
    "  • OSLinux-OSLinux_LOCALDISK...sda_DSKRea... (anomaly_count=1, earliest=1615013160, severity≈1.89)\n"
    "  • OSLinux-OSLinux_LOCALDISK...sda_DSKBps / DSKTps (severity ≈1.53–1.82)\n"
    "- MG01 also shows some disk I/O entries and CPU WIO with severity ≈1.31–1.50.\n"
    "  → Interpretation: disk I/O and related OS-level metrics on MG01/MG02 are elevated (several >1.3× p95). MG01 also shows an extreme TCP-FIN-WAIT spike (severity ~6.7) late in the window.\n\n"
    "Overall recommendation (prioritized):\n"
    "1. Investigate MG02 disk I/O activity and the increased row_count from dockerA1 → MG02 (trace and metric corroboration).\n"
    "2. Investigate the MG01 TCP FIN-WAIT spike (possible connection teardown/backlog issue) at ts 1615013880.\n"
    "3. Check related host-level OS metrics (I/O wait, disk throughput) on MG01/MG02 and the dockerA1 source for traffic patterns.\n\n"
    "Stored variables for drill-down: earliest_trace_anoms, mg_metric_highsev."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                     trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0    MG02  trace.from_dockerA1.row_count              2           1615012200         1615012920                104.0          79.55        1.307354,   cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value_in_window  p95_threshold  severity_ratio
0    MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              1           1615013880         1615013880               1.0000        0.15000        6.666667
1    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              1           1615013160         1615013160             872.0000      462.00000        1.887446
2    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1           1615013520         1615013520               4.0000        2.20000        1.818182
3    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead              1           1615013040         1615013040             852.4000      486.64000        1.751603
4    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps              1           1615013040         1615013040               6.0500        3.95750        1.528743
5    MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1           1615012800         1615012800               6.0000        4.00000        1.500000
6    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps              1           1615013160         1615013160               7.0000        4.75000        1.473684
7    MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1           1615012440         1615012440               6.0000        4.20000        1.428571
8    MG01                             OSLinux-CPU_CPU_CPUWio              1           1615012740         1615012740               2.0545        1.56848        1.309867)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: trace_anoms, trace_window
# Ensure timestamp and value types
trace_window["timestamp"] = pd.to_numeric(trace_window["timestamp"], errors="coerce").astype("Int64")
trace_window["value"] = pd.to_numeric(trace_window["value"], errors="coerce")

# Filter trace_anoms for MG01/MG02 and valid p95_threshold
ta = trace_anoms.copy()
ta_valid = ta[
    ta["cmdb_id"].isin(["MG01", "MG02"]) &
    ta["p95_threshold"].notna()
].copy()

results = []
# Helper to compute longest consecutive run (diff <= 60s)
def longest_run_from_sorted_ts(ts_list):
    if len(ts_list) == 0:
        return 0, pd.NA, pd.NA
    longest_len = 1
    best_start = ts_list[0]
    best_end = ts_list[0]
    cur_start = ts_list[0]
    cur_prev = ts_list[0]
    cur_len = 1
    for t in ts_list[1:]:
        if (t - cur_prev) <= 60:
            cur_len += 1
            cur_prev = t
        else:
            # end current run
            if cur_len > longest_len:
                longest_len = cur_len
                best_start = cur_start
                best_end = cur_prev
            # start new run
            cur_start = t
            cur_prev = t
            cur_len = 1
    # final check
    if cur_len > longest_len:
        longest_len = cur_len
        best_start = cur_start
        best_end = cur_prev
    return int(longest_len), int(best_start), int(best_end)

# Iterate groups
for _, row in ta_valid.iterrows():
    cmdb = row["cmdb_id"]
    tname = row["trace_name"]
    p95 = row["p95_threshold"]
    # select matching rows from trace_window
    grp = trace_window[(trace_window["cmdb_id"] == cmdb) & (trace_window["trace_name"] == tname)].copy()
    if grp.empty:
        continue
    # find anomaly timestamps where value >= p95
    anom_rows = grp[grp["value"] >= p95].copy()
    if anom_rows.empty:
        continue
    ts_sorted = sorted(anom_rows["timestamp"].dropna().astype(int).unique().tolist())
    # limit list to first 50 timestamps
    ts_limited = ts_sorted[:50]
    earliest_anom_ts = int(ts_sorted[0]) if len(ts_sorted)>0 else pd.NA
    total_anom_count = int(len(anom_rows))
    max_val_in_window = float(anom_rows["value"].max())
    # compute longest consecutive run
    longest_len, run_start, run_end = longest_run_from_sorted_ts(ts_sorted)
    results.append({
        "cmdb_id": cmdb,
        "trace_name": tname,
        "p95_threshold": float(p95),
        "max_value_in_window": max_val_in_window,
        "list_of_anomaly_timestamps": ts_limited,
        "earliest_anomaly_ts": earliest_anom_ts,
        "total_anomaly_count": total_anom_count,
        "longest_consecutive_run_length": longest_len,
        "longest_run_start_ts": run_start if not pd.isna(run_start) else pd.NA,
        "longest_run_end_ts": run_end if not pd.isna(run_end) else pd.NA
    })

# Build DataFrame
trace_fault_segments = pd.DataFrame(results)

# If empty, ensure consistent columns
expected_cols = ["cmdb_id","trace_name","p95_threshold","max_value_in_window","list_of_anomaly_timestamps",
                 "earliest_anomaly_ts","total_anomaly_count","longest_consecutive_run_length",
                 "longest_run_start_ts","longest_run_end_ts"]
for c in expected_cols:
    if c not in trace_fault_segments.columns:
        trace_fault_segments[c] = pd.NA

# Sort by longest run length desc then max_value_in_window desc
trace_fault_segments = trace_fault_segments.sort_values(
    ["longest_consecutive_run_length", "max_value_in_window"],
    ascending=[False, False]
).reset_index(drop=True)

# Keep full result in kernel variable trace_fault_segments (already assigned)
# Prepare compact top-20 view
trace_fault_segments_top20 = trace_fault_segments.head(20)[expected_cols]

# Display compact result (variable)
trace_fault_segments_top20

```
Out[7]:
```
Summary of MG01/MG02 trace fault segments (top 20 groups):

- Total groups returned: 20. All matching groups show anomaly events where trace value >= the group’s global p95.

Top groups by longest consecutive run:
1. MG02 / trace.from_Tomcat04.row_count
   - p95_threshold: 288.95, max_value_in_window: 305.0
   - anomaly timestamps: [1615013640, 1615013700]
   - earliest_anomaly_ts: 1615013640, total_anomaly_count: 2
   - longest consecutive run: 2 (from 1615013640 to 1615013700)

2. MG02 / trace.from_Tomcat01.row_count
   - p95_threshold: 277.30, max: 293.0
   - timestamps: [1615013820, 1615013880], longest run 2 (1615013820→1615013880)

3. MG02 / trace.from_dockerA2.row_count
   - p95_threshold: 78.20, max: 100.0
   - timestamps: [1615012860, 1615012920], longest run 2 (1615012860→1615012920)

4. MG01 / trace.from_dockerA1.row_count
   - p95_threshold: 83.55, max: 86.0
   - timestamps: [1615012860, 1615012920], longest run 2 (1615012860→1615012920)

5. MG01 / trace.from_dockerB1.duration_mean
   - p95_threshold: 0.180094, max: 0.216313
   - timestamps: [1615013040, 1615013100], longest run 2 (1615013040→1615013100)

Other notable groups:
- Several trace.self.row_count groups (MG02 and MG01) exceed very large p95 thresholds (p95 ~1754.7 and 1718.5) with max values ~1902 and ~1746 respectively; each has 2 anomaly timestamps but longest run length = 1.
- High row_count spikes observed for dockerB-related flows:
  - MG02 trace.from_dockerB1.row_count max = 349.0 (two anomaly timestamps)
  - MG01 trace.from_dockerB1.row_count max = 332.0
  - MG02 trace.to_dockerB2.row_count max = 337.0
- One group has 3 anomalies: MG01 / trace.from_Tomcat03.row_count
  - timestamps: [1615012200, 1615012380, 1615012920], total_anomaly_count = 3 (but longest consecutive run = 1)

Interpretation and next actions:
- Prioritize investigation of flows with the longest consecutive runs (run length = 2) and high max values, especially:
  - MG02: from_Tomcat04, from_Tomcat01, from_dockerA2, from_dockerB1, to_dockerB2.
  - MG01: from_dockerA1, from_dockerB1, to_dockerB2.
- Also review trace.self.row_count spikes on MG02/MG01 (very high counts) and the MG01 group with 3 anomaly hits (from_Tomcat03).
- Use the listed anomaly timestamps (numeric seconds) to correlate with system logs, metrics, and GC spikes for root-cause triage.

Stored variable for further analysis: trace_fault_segments (full result); the summary above is based on its top 20 entries.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  p95_threshold  max_value_in_window            list_of_anomaly_timestamps  earliest_anomaly_ts  total_anomaly_count  longest_consecutive_run_length  longest_run_start_ts  longest_run_end_ts
0     MG02      trace.from_Tomcat04.row_count     288.950000           305.000000              [1615013640, 1615013700]           1615013640                    2                               2            1615013640          1615013700
1     MG02      trace.from_Tomcat01.row_count     277.300000           293.000000              [1615013820, 1615013880]           1615013820                    2                               2            1615013820          1615013880
2     MG02      trace.from_dockerA2.row_count      78.200000           100.000000              [1615012860, 1615012920]           1615012860                    2                               2            1615012860          1615012920
3     MG01      trace.from_dockerA1.row_count      83.550000            86.000000              [1615012860, 1615012920]           1615012860                    2                               2            1615012860          1615012920
4     MG01  trace.from_dockerB1.duration_mean       0.180094             0.216313              [1615013040, 1615013100]           1615013040                    2                               2            1615013040          1615013100
5     MG02               trace.self.row_count    1754.700000          1902.000000              [1615012380, 1615012920]           1615012380                    2                               1            1615012380          1615012380
6     MG01               trace.self.row_count    1718.500000          1746.000000              [1615012380, 1615012920]           1615012380                    2                               1            1615012380          1615012380
7     MG02      trace.from_dockerB1.row_count     306.100000           349.000000              [1615012380, 1615012500]           1615012380                    2                               1            1615012380          1615012380
8     MG02        trace.to_dockerB2.row_count     304.250000           337.000000              [1615013040, 1615013880]           1615013040                    2                               1            1615013040          1615013040
9     MG01      trace.from_dockerB1.row_count     291.750000           332.000000              [1615012380, 1615012500]           1615012380                    2                               1            1615012380          1615012380
10    MG01        trace.to_dockerB2.row_count     283.550000           327.000000              [1615012740, 1615013880]           1615012740                    2                               1            1615012740          1615012740
11    MG01      trace.from_dockerB2.row_count     293.400000           321.000000              [1615012380, 1615012500]           1615012380                    2                               1            1615012380          1615012380
12    MG02        trace.to_dockerB1.row_count     300.650000           314.000000              [1615013640, 1615013880]           1615013640                    2                               1            1615013640          1615013640
13    MG02      trace.from_Tomcat03.row_count     276.500000           310.000000              [1615012920, 1615013940]           1615012920                    2                               1            1615012920          1615012920
14    MG02      trace.from_dockerB2.row_count     289.400000           303.000000              [1615012380, 1615012500]           1615012380                    2                               1            1615012380          1615012380
15    MG01        trace.to_dockerB1.row_count     291.850000           303.000000              [1615013040, 1615013880]           1615013040                    2                               1            1615013040          1615013040
16    MG01      trace.from_Tomcat02.row_count     278.950000           293.000000              [1615012980, 1615013880]           1615012980                    2                               1            1615012980          1615012980
17    MG02      trace.from_Tomcat02.row_count     281.650000           284.000000              [1615012200, 1615013040]           1615012200                    2                               1            1615012200          1615012200
18    MG01      trace.from_Tomcat04.row_count     276.550000           283.000000              [1615012380, 1615013880]           1615012380                    2                               1            1615012380          1615012380
19    MG01      trace.from_Tomcat03.row_count     277.000000           281.000000  [1615012200, 1615012380, 1615012920]           1615012200                    3                               1            1615012200          1615012200```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import datetime

# Reuse kernel variables: trace_fault_segments, trace_anoms, metric_anoms, component_metric_summary
# Ensure they exist
for var in ["trace_fault_segments", "trace_anoms", "metric_anoms"]:
    if var not in globals():
        raise RuntimeError(f"Required variable '{var}' not found in kernel.")

# Allowed components
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Prepare lookups
tfs = trace_fault_segments.copy()
ta = trace_anoms.copy()
ma = metric_anoms.copy()

# Ensure numeric types
tfs["longest_run_start_ts"] = pd.to_numeric(tfs["longest_run_start_ts"], errors="coerce").astype("Int64")
tfs["longest_run_end_ts"] = pd.to_numeric(tfs["longest_run_end_ts"], errors="coerce").astype("Int64")
tfs["longest_consecutive_run_length"] = pd.to_numeric(tfs["longest_consecutive_run_length"], errors="coerce").fillna(0).astype(int)
tfs["p95_threshold"] = pd.to_numeric(tfs["p95_threshold"], errors="coerce")
tfs["max_value_in_window"] = pd.to_numeric(tfs["max_value_in_window"], errors="coerce")

ta["severity_ratio"] = pd.to_numeric(ta["severity_ratio"], errors="coerce")
ta["p95_threshold"] = pd.to_numeric(ta["p95_threshold"], errors="coerce")

ma["severity_ratio"] = pd.to_numeric(ma["severity_ratio"], errors="coerce")
ma["earliest_anomaly_ts"] = pd.to_numeric(ma["earliest_anomaly_ts"], errors="coerce").astype("Int64")
ma["p95_threshold"] = pd.to_numeric(ma["p95_threshold"], errors="coerce")
ma["max_value_in_window"] = pd.to_numeric(ma["max_value_in_window"], errors="coerce")

# Build candidate rows
candidates = []
tz = pytz.timezone("Asia/Shanghai")

for comp in allowed_components:
    # a) trace segments with run >=2
    comp_tfs = tfs[(tfs["cmdb_id"] == comp) & (tfs["longest_consecutive_run_length"] >= 2)].copy()
    if comp_tfs.empty:
        continue
    # earliest run start ts among those segments
    earliest_run_ts = int(comp_tfs["longest_run_start_ts"].min())
    # collect trace evidence: include severity by joining with trace_anoms (if present)
    # merge to get severity (may have duplicates, use first)
    merged = pd.merge(
        comp_tfs,
        ta[["cmdb_id","trace_name","p95_threshold","severity_ratio"]].rename(columns={"p95_threshold":"p95_from_anoms","severity_ratio":"trace_severity"}),
        how="left",
        on=["cmdb_id","trace_name"]
    )
    # Build evidence strings up to, say, 10 traces
    trace_evidence_items = []
    max_trace_severity = 0.0
    for _, r in merged.sort_values(["longest_consecutive_run_length","max_value_in_window"], ascending=[False, False]).iterrows():
        tname = r["trace_name"]
        start = int(r["longest_run_start_ts"]) if pd.notna(r["longest_run_start_ts"]) else None
        end = int(r["longest_run_end_ts"]) if pd.notna(r["longest_run_end_ts"]) else None
        maxval = r["max_value_in_window"]
        p95 = r.get("p95_threshold", r.get("p95_from_anoms", pd.NA))
        sev = r.get("trace_severity", pd.NA)
        try:
            sev_f = float(sev) if pd.notna(sev) else None
        except:
            sev_f = None
        if sev_f is not None and sev_f > max_trace_severity:
            max_trace_severity = sev_f
        item = f"{tname}: {start}→{end}, max={maxval}, p95={p95}, sev={sev_f}"
        trace_evidence_items.append(item)
    trace_evidence_str = "; ".join(trace_evidence_items[:10])  # limit length

    # b) find metric_anoms for same component with severity >=1.3 and earliest_anomaly_ts within ±300s of earliest_run_ts
    window_start = earliest_run_ts - 300
    window_end = earliest_run_ts + 300
    comp_metrics = ma[
        (ma["cmdb_id"] == comp) &
        (ma["severity_ratio"] >= 1.3) &
        (ma["earliest_anomaly_ts"].notna()) &
        (ma["earliest_anomaly_ts"] >= window_start) &
        (ma["earliest_anomaly_ts"] <= window_end)
    ].copy()
    metric_evidence_count = int(len(comp_metrics))
    # prepare up to 5 example strings
    metric_examples = []
    for _, rm in comp_metrics.sort_values(["severity_ratio","anomaly_count"], ascending=[False,False]).head(5).iterrows():
        k = rm["kpi_name"]
        ets = int(rm["earliest_anomaly_ts"]) if pd.notna(rm["earliest_anomaly_ts"]) else None
        sev = float(rm["severity_ratio"]) if pd.notna(rm["severity_ratio"]) else None
        metric_examples.append(f"{k} ({ets}, sev={sev})")
    metric_examples_str = "; ".join(metric_examples)

    # convert earliest_run_ts to datetime UTC+8
    dt_local = datetime.datetime.fromtimestamp(earliest_run_ts, tz)

    candidates.append({
        "component": comp,
        "earliest_run_ts": earliest_run_ts,
        "earliest_run_ts_as_datetime_UTC+8": dt_local.strftime("%Y-%m-%d %H:%M:%S"),
        "trace_evidence": trace_evidence_str,
        "max_trace_severity": max_trace_severity,
        "metric_evidence_count": metric_evidence_count,
        "metric_examples": metric_examples_str
    })

# Build DataFrame
root_cause_candidates = pd.DataFrame(candidates)

# If empty, ensure columns exist
if root_cause_candidates.empty:
    root_cause_candidates = pd.DataFrame(columns=[
        "component","earliest_run_ts","earliest_run_ts_as_datetime_UTC+8",
        "trace_evidence","max_trace_severity","metric_evidence_count","metric_examples"
    ])
else:
    # Sort by metric_evidence_count desc then by max_trace_severity desc
    root_cause_candidates = root_cause_candidates.sort_values(
        ["metric_evidence_count","max_trace_severity"],
        ascending=[False,False]
    ).reset_index(drop=True)

# Limit to top 10
root_cause_candidates = root_cause_candidates.head(10)

# Final compact table (stored in kernel variable as required)
root_cause_candidates

```
Out[8]:
```
Summary of candidate root causes (concise):

- Two top candidate components: MG02 and MG01. Both have trace fault segments with consecutive runs (longest_consecutive_run_length >= 2) whose earliest correlated start time is 1615012860 (2021-03-06 14:41:00 UTC+8).

MG02 (strongest candidate)
- Earliest trace-run start: 1615012860 → 2021-03-06 14:41:00 (UTC+8).
- Trace evidence (examples): trace.from_Tomcat04.row_count (run 1615013640→1615013700, max ~305), trace.from_Tomcat01.row_count (1615013820→1615013880, max ~293), trace.from_dockerA2.row_count (1615012860→1615012920, max ~100), plus several dockerB/Tomcat row_count traces.
- Metric evidence count: 4 supporting metric KPIs within ±5 minutes of the earliest trace run.
- Metric examples: multiple local-disk IO KPIs on MG02 (e.g., sda DSKRead/DSKBps/DSKTps) exceeding their p95 by ~1.5–1.9×.
- Interpretation: trace spikes involving Tomcat and dockerA/dockerB flows coincide with elevated disk I/O metrics on MG02. This suggests MG02 disk I/O pressure is a likely contributor to observed latency on those flows.

MG01 (secondary candidate)
- Earliest trace-run start: 1615012860 → 2021-03-06 14:41:00 (UTC+8).
- Trace evidence (examples): trace.from_dockerA1.row_count (1615012860→1615012920, max ~86), trace.from_dockerB1.duration_mean/run pairs.
- Metric evidence count: 2 supporting metric KPIs within ±5 minutes.
- Metric examples: MG01 local-disk I/O KPI(s) and OS-level I/O wait / DSKBps exceed p95 by ~1.3–1.5× (and CPU WIO shows elevation).
- Interpretation: MG01 shows disk I/O / I/O-wait elevations temporally correlated with dockerA/dockerB→MG flows, implicating MG01 I/O as a contributing factor.

Recommended next steps (prioritized)
1. Investigate MG02 disk subsystem: check disk throughput, I/O queueing, and processes causing DSKRead/DSKBps spikes around 14:41 and subsequent anomaly timestamps.
2. Correlate network/trace sources: inspect dockerA1/dockerA2 and Tomcat nodes producing the high row_count / duration traces into MG02 and MG01.
3. On MG01, review I/O wait and disk metrics plus the later TCP FIN-WAIT spike to see if connection handling/backlog issues co-occur.
4. Use the listed anomaly timestamps to align logs, GC events, and system metrics for root-cause confirmation.

Stored result for drill-down: root_cause_candidates (compact table linking trace runs and supporting metric KPIs).

The original code execution output of IPython Kernel is also provided below for reference:

component  earliest_run_ts earliest_run_ts_as_datetime_UTC+8                                     trace_evidence  max_trace_severity  metric_evidence_count                                    metric_examples
0      MG02       1615012860               2021-03-06 14:41:00  trace.from_Tomcat04.row_count: 1615013640→1615...            1.278772                      4  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...
1      MG01       1615012860               2021-03-06 14:41:00  trace.from_dockerA1.row_count: 1615012860→1615...            1.201114                      2  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...```
```